# 第7章：建立聊天應用程式
## Microsoft Foundry 模型 API 快速入門

本筆記本改編自包含可存取 [Azure OpenAI](notebook-azure-openai.ipynb) 服務筆記本的 [Azure OpenAI 範例資源庫](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst)。

> **注意：** GitHub Models 將於2026年7月底退役。本筆記本現針對提供相同免費試用模型目錄及 Azure AI 推論 SDK 體驗的 [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)。


# 概述  
「大型語言模型是將文字映射到文字的函數。給定一個輸入文字字串，大型語言模型會嘗試預測接下來會出現的文字」(1)。這個「快速入門」筆記本將向使用者介紹大型語言模型的高階概念、開始使用 AML 所需的核心套件需求、提示設計的簡要介紹，以及幾個不同使用案例的短範例。 


## 目錄  

[概觀](#overview)  
[如何使用 OpenAI 服務](#how-to-use-openai-service)  
[1. 建立您的 OpenAI 服務](#1.-creating-your-openai-service)  
[2. 安裝](#2.-installation)    
[3. 認證資訊](#3.-credentials)  

[使用案例](#use-cases)    
[1. 摘要文字](#1.-summarize-text)  
[2. 分類文字](#2.-classify-text)  
[3. 產生新產品名稱](#3.-generate-new-product-names)  
[4. 微調分類器](#4.fine-tune-a-classifier)  

[參考資料](#references)


### 建立你的第一個提示  
此簡短練習將為使用 Microsoft Foundry Models 向模型提交提示，以完成簡單任務「摘要」提供基本介紹。  


<strong>步驟</strong>：  
1. 如果尚未安裝，請在你的 Python 環境中安裝 `azure-ai-inference` 函式庫。  
2. 載入標準輔助函式庫並設定你的 Microsoft Foundry Models 認證。  
3. 選擇適合你的任務的模型  
4. 為模型建立簡單的提示  
5. 向模型 API 提交你的請求！  


### 1. 安裝 `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. 匯入輔助程式庫並實例化憑證


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. 找到合適的模型  
像 GPT-4o 和 GPT-4o mini 這樣的模型能夠理解和生成自然語言，並且在 Microsoft Foundry Models 目錄中與來自 Meta、Mistral、Cohere 和 Microsoft 的模型一起提供。


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. 提示設計  

「大型語言模型的魔力在於，通過在大量文本中訓練以最小化這種預測誤差，模型最終學會了對這些預測有用的概念。例如，它們學會了像是」(1):

* 如何拼寫
* 語法如何運作
* 如何改寫
* 如何回答問題
* 如何進行對話
* 如何用多種語言寫作
* 如何編程
* 等等。

#### 如何控制大型語言模型  
「在所有輸入給大型語言模型的資料中，最有影響力的無疑是文本提示」(1)。

大型語言模型可以通過幾種方式被提示以產生輸出：

指令：告訴模型你想要什麼
補全：引導模型完成你想要的開頭部分
示範：向模型展示你想要的內容，可以是：
提示中包含幾個示例
有數百或數千個示例的微調訓練數據集」



#### 創建提示的三條基本指南：

<strong>顯示並說明</strong>。透過指令、示例或兩者結合，清楚表達你想要什麼。如果你想讓模型按字母順序排列一個項目列表或按情感分類一段文字，就要明確告訴它這是你的需求。

<strong>提供優質數據</strong>。如果你想建立分類器或讓模型遵循某個模式，確保有足夠的示例。一定要校對你的示例——模型通常足夠聰明，可以洞察基本拼寫錯誤並回應你，但它也可能假設這是故意的，這會影響回應。

**檢查你的設定。** 溫度和 top_p 設定控制模型生成回應的決定性。如果你要求模型提供唯一正確答案的回應，那麼你會希望將這些設置調低。如果你想要更多樣化的回應，則可能想把它們調高。人們在這些設定上最常犯的錯誤是以為它們是「聰明度」或「創意」的控制。


資料來源: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. 送出！


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### 重複相同的呼叫，結果比較如何？ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## 摘要文本  
#### 挑戰  
將摘要文本加上「tl;dr:」放在文本段落的結尾。注意模型如何在沒有額外指示的情況下執行多種任務。您可以嘗試使用比 tl;dr 更具描述性的提示來修改模型行為並自訂摘要內容(3)。  

近期研究顯示，先在大量文本語料庫上進行預訓練，接著針對特定任務的微調，能在許多自然語言處理任務與基準測試中取得顯著成效。雖然架構通常不依賴任務，但此方法仍需數千或數萬範例的任務專用微調數據集。相比之下，人類通常只需少數範例或簡單指令便能完成新的語言任務，而目前的自然語言處理系統仍大多無法達成此點。我們在此展示放大語言模型規模，可大幅提升任務無關的少量範例學習表現，有時甚至能達到先前最先進微調方法的競爭力。 



摘要  


# 多種使用情境練習  
1. 摘要文本  
2. 分類文本  
3. 產生新產品名稱


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 分類文本  
#### 挑戰  
在推理時將項目分類到提供的類別中。在以下示例中，我們在提示中同時提供了類別和要分類的文本(*playground_reference)。 

客戶詢問：您好，我的筆記型電腦鍵盤上的一個按鍵最近壞了，我需要更換：

分類類別：


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## 產生新產品名稱
#### 挑戰
根據範例字詞創造產品名稱。在提示中，我們加入了關於將要生成名稱的產品資訊。我們也提供了一個類似範例來展示我們希望得到的模式。我們還將溫度值設定為較高，以提高隨機性並產生更具創新性的回應。

產品描述：一款家用奶昔機
種子詞：快速、健康、緊湊。
產品名稱：HomeShaker、Fit Shaker、QuickShake、Shake Maker

產品描述：一雙可以適合任何腳尺寸的鞋子。
種子詞：適應性、合腳、全方位合腳。


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# 參考資料  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry 入口網站](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [針對 GPT 模型微調以分類文本的最佳實務](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# 需要更多協助  
[OpenAI 商業化團隊](AzureOpenAITeam@microsoft.com) 


# 貢獻者
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
